# Calibration Loss Development Notebook (Debug + Best-Practice Search)

This notebook is the dedicated **development and debugging workspace** for calibration-loss training in ANCOVA NPE.

It has two goals:
1. Isolate and debug calibration-loss failure modes (scale mismatch, Lagrangian dual dynamics, instability).
2. Systematically search for robust hyperparameter settings and document a best-practice recipe.

Use this notebook to evolve settings; keep `ancova_calibration_loss.ipynb` focused on clean baseline-vs-calibrated comparison.

## Workflow in This Notebook

- **Phase 1 — Sanity checks:** Validate package API + prior scale behavior.
- **Phase 2 — Coarse sweep:** Explore Lagrangian-control and calibration hyperparameters quickly.
- **Phase 3 — Refinement:** Re-run best candidates with larger budgets and multiple seeds.
- **Phase 4 — Recommendation:** Emit one best-practice settings dictionary for downstream notebooks.

In [1]:
%pip install -e ".."
%pip install -e "../../bfcalloss"

import os
import sys
import traceback
import inspect
from dataclasses import asdict

if not os.environ.get("KERAS_BACKEND"):
    os.environ["KERAS_BACKEND"] = "torch"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import keras
import bayesflow as bf
import bayesflow_calibration as bfc

from rctbp_bf_training.models.ancova.model import (
    ANCOVAConfig,
    create_ancova_adapter,
    create_ancova_workflow_components,
    create_simulator,
    create_validation_grid,
    make_simulate_fn,
)
from rctbp_bf_training.core.utils import sample_t_or_normal
from rctbp_bf_training.core.validation import (
    run_validation_pipeline,
    make_bayesflow_infer_fn,
)

from bayesflow_calibration import (
    CalibratedContinuousApproximator,
    CalibrationMonitorCallback,
    suggest_calibration_settings,
    print_memory_report,
)

required_calibration_kwargs = (
    "target_calibration_error",
    "lr_lambda",
    "lambda_max",
    "normalization_burn_in",
    "batch_size_calibration",
)
init_params = inspect.signature(CalibratedContinuousApproximator.__init__).parameters
missing_kwargs = [name for name in required_calibration_kwargs if name not in init_params]
if missing_kwargs:
    raise RuntimeError(
        "Loaded bayesflow_calibration does not match expected Lagrangian API. "
        f"Missing __init__ kwargs: {missing_kwargs}. "
        "Reinstall package and restart kernel before continuing."
    )

print(f"Python executable: {sys.executable}")
print(f"bayesflow_calibration source: {bfc.__file__}")
print(f"BayesFlow {bf.__version__}")
print(f"Keras {keras.__version__} (backend: {keras.backend.backend()})")
print("Lagrangian calibration API check passed")

Obtaining file:///C:/Users/Matze/Documents/GitHub/rctbp_bf_training
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for rctbp_bf_training (pyproject.toml): started
  Building editable for rctbp_bf_training (pyproject.toml): finished with status 'done'
  Created wheel for rctbp_bf_training: filename=rctbp_bf_training-0.1.0-0.editable-py3-none-any.whl size=2700 sha256=d2459c87dae26a4fb171d1e9b715fc8c17b553e23cb9883fde971d5d713818f8
  Stored in directory: C:\Users\Matze\AppData\Local\Temp\pip-ephem-wheel-cache-0qwi3_lw\wheels\db\


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Obtaining file:///C:/Users/Matze/Documents/GitHub/bfcalloss
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for bayesflow-calibration-loss (pyproject.toml): started
  Building editable for bayesflow-calibration-loss (pyproject.toml): finished with status 'done'
  Created wheel for bayesflow-calibration-loss: filename=bayesflow_calibration_loss-0.1.0-0.editable-py3-none-any.whl size=3881 sha256=14166ceb8f3e78b66f37b6ee73b3cd1252745cd202c5becc0d98a70488b20812
  Stored in directory: C:\Users\Matze\AppData\Local\Temp\pip-ephem-whe

INFO:bayesflow:Using backend 'torch'
When using torch backend, we need to disable autograd by default to avoid excessive memory usage. Use

with torch.enable_grad():
    ...

in contexts where you need gradients (e.g. custom training loops).


ModuleNotFoundError: No module named 'bayesflow_calibration'

In [ ]:
GLOBAL_SEED = 2026
RNG = np.random.default_rng(GLOBAL_SEED)

config = ANCOVAConfig()
config.workflow.summary_network.summary_dim = 6
config.workflow.summary_network.depth = 2
config.workflow.summary_network.width = 32
config.workflow.summary_network.dropout = 0.05
config.workflow.inference_network.depth = 4
config.workflow.inference_network.hidden_sizes = (64, 64)
config.workflow.inference_network.dropout = 0.10

simulator = create_simulator(config, RNG)
adapter = create_ancova_adapter()
train_cfg = config.workflow.training

def make_optimizer(batch_size, batches_per_epoch):
    steps_per_epoch = batch_size * batches_per_epoch
    lr_schedule = keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=train_cfg.initial_lr,
        decay_steps=steps_per_epoch,
        decay_rate=train_cfg.decay_rate,
        staircase=True,
    )
    return keras.optimizers.Adam(learning_rate=lr_schedule)

def build_workflow(batch_size, batches_per_epoch):
    summary_net, inference_net, _ = create_ancova_workflow_components(config)
    workflow = bf.BasicWorkflow(
        simulator=simulator,
        adapter=adapter,
        inference_network=inference_net,
        summary_network=summary_net,
        optimizer=make_optimizer(batch_size, batches_per_epoch),
        inference_conditions=["N", "p_alloc", "prior_df", "prior_scale"],
    )
    return workflow, summary_net, inference_net

print("Shared components initialized")
print("Summary network:", config.workflow.summary_network)
print("Inference network:", config.workflow.inference_network)

# Meta-parameter batching sanity check.
# We inspect both:
# 1) variation within one simulator.sample(...) call
# 2) variation across repeated sample(...) calls
BATCH_META_CHECK_SIMS = 512
meta_check = simulator.sample(BATCH_META_CHECK_SIMS)

df_within = np.asarray(meta_check["prior_df"]).reshape(-1)
scale_within = np.asarray(meta_check["prior_scale"]).reshape(-1)
unique_prior_df_within = int(np.unique(df_within).size)
unique_prior_scale_within = int(np.unique(np.round(scale_within, 6)).size)

print("\nMeta-parameter batching check:")
print(f"  sims per call: {BATCH_META_CHECK_SIMS}")
print(f"  unique prior_df within call: {unique_prior_df_within}")
print(f"  unique prior_scale within call: {unique_prior_scale_within}")

if unique_prior_df_within > 1 and unique_prior_scale_within > 1:
    print("  ✓ Meta parameters vary within a sampled batch.")
else:
    print("  ! Meta parameters appear shared within a single sample() call.")
    print("    Checking variation across repeated sample() calls...")

    n_calls = 32
    df_across = []
    scale_across = []
    for _ in range(n_calls):
        draw = simulator.sample(BATCH_META_CHECK_SIMS)
        df_across.append(float(np.asarray(draw["prior_df"]).reshape(-1)[0]))
        scale_across.append(float(np.asarray(draw["prior_scale"]).reshape(-1)[0]))

    unique_df_across = int(np.unique(np.asarray(df_across)).size)
    unique_scale_across = int(np.unique(np.round(np.asarray(scale_across), 6)).size)

    print(f"  unique prior_df across {n_calls} calls: {unique_df_across}")
    print(f"  unique prior_scale across {n_calls} calls: {unique_scale_across}")

    if unique_df_across > 1 and unique_scale_across > 1:
        print("  ✓ Meta parameters vary across batches (training still sees varying context over epochs).")
    else:
        raise RuntimeError(
            "Meta parameters did not vary across repeated simulator calls. "
            "Posterior conditioning on meta context is unlikely to be learnable."
        )

In [ ]:
import gc

# ── Memory profiling ──────────────────────────────────────────────────────────
# Builds a temporary inference network (same architecture as the sweep) to:
#   1. Print an analytical/empirical memory estimate table so you can see the
#      cost of each (subsample_size, n_samples) combination before training.
#   2. Auto-select subsample_size, n_samples, and log_prob_chunk_size that fit
#      safely within the available GPU (or CPU-fallback) memory budget.
#
# The resulting MEMORY_SETTINGS object is read by the run_variant wrapper below
# to replace None subsample sizes, cap excessive n_samples, and enable chunked
# log_prob evaluation when needed.

_tmp_summary_net, _tmp_inf_net, _ = create_ancova_workflow_components(config)
_tmp_sample = simulator.sample(4)
_tmp_adapted = adapter(_tmp_sample)

_PARAM_DIM = int(np.asarray(_tmp_adapted["inference_variables"]).shape[-1])
_COND_DIM = (
    config.workflow.summary_network.summary_dim
    + int(np.asarray(_tmp_adapted["inference_conditions"]).shape[-1])
)

print(f"Calibration memory estimate (param_dim={_PARAM_DIM}, cond_dim={_COND_DIM}):")
print_memory_report(
    _tmp_inf_net,
    param_dim=_PARAM_DIM,
    cond_dim=_COND_DIM,
    subsample_sizes=(64, 128, 192, 256, 384, 512, 1024),
    n_samples_list=(100, 200, 500, 1000),
)

MEMORY_SETTINGS = suggest_calibration_settings(
    _tmp_inf_net,
    param_dim=_PARAM_DIM,
    cond_dim=_COND_DIM,
)

print(f"\nAuto-selected calibration settings:")
print(f"  subsample_size      = {MEMORY_SETTINGS.subsample_size}")
print(f"  n_samples           = {MEMORY_SETTINGS.n_samples}")
print(f"  log_prob_chunk_size = {MEMORY_SETTINGS.log_prob_chunk_size}")
print(f"  estimated_peak_mb   = {MEMORY_SETTINGS.estimated_peak_mb:.1f} MB")
print(f"  headroom_pct        = {MEMORY_SETTINGS.headroom_pct:.0f}%")

del _tmp_summary_net, _tmp_inf_net, _tmp_sample, _tmp_adapted
keras.backend.clear_session()
gc.collect()

## Phase 1: Coarse Hyperparameter Sweep

This coarse sweep intentionally uses short training budgets to detect stable regions quickly.

Ranking priority:
1. Lower calibration error
2. No severe NRMSE degradation
3. Stable optimization (no crashes / NaN losses)

In [ ]:
COARSE_EPOCHS = 20
COARSE_BATCH_SIZE = 256
COARSE_BATCHES_PER_EPOCH = 50
COARSE_VAL_SIMS = 500
COARSE_TEST_SIMS = 500

# Coarse condition-grid evaluation budget (kept modest for sweep runtime).
COARSE_GRID_N_SIMS = 200
COARSE_GRID_N_POST_DRAWS = 200
COARSE_GRID_EXTENDED = False

# Default Lagrangian settings for the new calibration API.
DEFAULT_TARGET_CALIBRATION_ERROR = 0.02
DEFAULT_LR_LAMBDA = 0.05

def build_schedule(kind, gamma_max, start_epoch, end_epoch):
    if kind not in {"windowed_cosine", "windowed_linear", "step"}:
        raise ValueError(f"Unknown schedule kind: {kind}")
    return {
        "kind": str(kind),
        "start_epoch": int(start_epoch),
        "end_epoch": int(end_epoch),
        "lambda_max": float(gamma_max),
    }

def _safe_float(value):
    try:
        if value is None:
            return np.nan
        if hasattr(value, "numpy"):
            value = value.numpy()
        arr = np.asarray(value).reshape(-1)
        if arr.size == 0:
            return np.nan
        scalar = float(arr[0])
        return scalar if np.isfinite(scalar) else np.nan
    except Exception:
        return np.nan

class SafeCalibrationMetricsCallback(keras.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.last_calibration_loss = np.nan
        self.last_cal_ema = np.nan
        self.last_gamma = np.nan
        self.last_lambda = np.nan

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        model = getattr(self, "model", None)
        if model is None:
            return

        self.last_cal_ema = _safe_float(logs.get("cal_ema", logs.get("calibration_ema", np.nan)))
        self.last_gamma = _safe_float(logs.get("gamma", np.nan))
        self.last_lambda = _safe_float(logs.get("lambda", getattr(model, "_lambda", np.nan)))

        raw_calibration = _safe_float(
            logs.get("calibration_loss", logs.get("calibration_error", np.nan))
        )
        if not np.isfinite(raw_calibration):
            raw_calibration = self.last_cal_ema
        self.last_calibration_loss = raw_calibration

        logs["calibration_loss"] = raw_calibration
        logs["calibration_ema"] = self.last_cal_ema
        logs["gamma"] = self.last_gamma
        logs["lambda"] = self.last_lambda

def extract_metric(metrics_obj, metric_name):
    if isinstance(metrics_obj, dict):
        val = metrics_obj.get(metric_name, np.nan)
        return float(val) if np.isfinite(val) else np.nan

    if isinstance(metrics_obj, pd.DataFrame):
        if metrics_obj.empty:
            return np.nan

        # Common BayesFlow layout: metrics on index, variables on columns
        if metric_name in metrics_obj.index:
            row = metrics_obj.loc[metric_name]
            if isinstance(row, pd.Series):
                vals = pd.to_numeric(row, errors="coerce").to_numpy(dtype=float)
                finite_vals = vals[np.isfinite(vals)]
                return float(finite_vals.mean()) if finite_vals.size else np.nan
            return float(row) if np.isfinite(row) else np.nan

        # Fallback: metric encoded in columns
        exact_cols = [c for c in metrics_obj.columns if c == metric_name]
        partial_cols = [c for c in metrics_obj.columns if metric_name.lower() in str(c).lower()]
        candidate_cols = exact_cols or partial_cols
        if not candidate_cols:
            return np.nan
        col_values = pd.to_numeric(metrics_obj[candidate_cols[0]], errors="coerce").to_numpy(dtype=float)
        finite_vals = col_values[np.isfinite(col_values)]
        return float(finite_vals[0]) if finite_vals.size else np.nan

    if isinstance(metrics_obj, (list, tuple)) and len(metrics_obj) > 0:
        first = metrics_obj[0]
        return extract_metric(first, metric_name)

    return np.nan

def run_variant(name, use_calibration, gamma_max=0.0, calibration_mode=np.nan,
               schedule_kind="windowed_cosine", start_epoch=10, end_epoch=15,
               n_samples=200, subsample_size=None, normalize_loss=True,
               ema_momentum=0.99, seed=2026,
               log_prob_chunk_size=None,
               epochs=COARSE_EPOCHS, batch_size=COARSE_BATCH_SIZE,
               batches_per_epoch=COARSE_BATCHES_PER_EPOCH,
               val_sims=COARSE_VAL_SIMS, test_sims=COARSE_TEST_SIMS):
    np.random.seed(seed)
    keras.utils.set_random_seed(seed)

    try:
        workflow, summary_net, inference_net = build_workflow(batch_size, batches_per_epoch)

        callback_list = []
        cal_metrics_cb = None
        if use_calibration:
            schedule = build_schedule(schedule_kind, gamma_max, start_epoch, end_epoch)
            active_window_epochs = max(1, schedule["end_epoch"] - schedule["start_epoch"] + 1)
            normalization_burn_in = max(20, int(0.5 * active_window_epochs * batches_per_epoch))

            approximator = CalibratedContinuousApproximator(
                inference_network=inference_net,
                summary_network=summary_net,
                adapter=adapter,
                start_epoch=int(schedule["start_epoch"]),
                target_calibration_error=float(DEFAULT_TARGET_CALIBRATION_ERROR),
                lr_lambda=float(DEFAULT_LR_LAMBDA),
                lambda_max=max(1e-6, float(schedule["lambda_max"])),
                normalize_loss=bool(normalize_loss),
                normalization_burn_in=int(normalization_burn_in),
                calibration_mode=float(calibration_mode),
                n_samples=int(n_samples),
                batch_size_calibration=None if subsample_size is None else int(subsample_size),
                log_prob_chunk_size=log_prob_chunk_size,
            )
            approximator.compile(optimizer=make_optimizer(batch_size, batches_per_epoch))
            workflow.approximator = approximator
            cal_metrics_cb = SafeCalibrationMetricsCallback()
            callback_list = [CalibrationMonitorCallback(), cal_metrics_cb]
        else:
            workflow.approximator.compile(optimizer=make_optimizer(batch_size, batches_per_epoch))

        history = workflow.fit_online(
            epochs=epochs,
            batch_size=batch_size,
            num_batches_per_epoch=batches_per_epoch,
            validation_data=val_sims,
            callbacks=callback_list,
        )

        test_data = simulator.sample(test_sims)
        diagnostics = workflow.compute_default_diagnostics(
            test_data=test_data,
            as_data_frame=True,
        )

        nrmse = extract_metric(diagnostics, "NRMSE")
        cal_error = extract_metric(diagnostics, "Calibration Error")
        contraction = extract_metric(diagnostics, "Posterior Contraction")

        if (not np.isfinite(nrmse)) or (not np.isfinite(cal_error)) or (not np.isfinite(contraction)):
            try:
                diagnostics_alt = workflow.compute_default_diagnostics(
                    test_data=test_data,
                    variable_keys=["inference_variables"],
                    variable_names=["b_group"],
                    as_data_frame=True,
                )
                nrmse_alt = extract_metric(diagnostics_alt, "NRMSE")
                cal_error_alt = extract_metric(diagnostics_alt, "Calibration Error")
                contraction_alt = extract_metric(diagnostics_alt, "Posterior Contraction")

                if np.isfinite(nrmse_alt):
                    nrmse = nrmse_alt
                if np.isfinite(cal_error_alt):
                    cal_error = cal_error_alt
                if np.isfinite(contraction_alt):
                    contraction = contraction_alt
            except Exception:
                pass

        # Condition-grid validation (aligned with ancova_calibration_loss.ipynb).
        grid_mean_cal_error = np.nan
        grid_overall_nrmse = np.nan
        grid_coverage_90 = np.nan
        grid_coverage_95 = np.nan
        grid_eval_error = ""
        try:
            conditions = create_validation_grid(extended=COARSE_GRID_EXTENDED)
            simulate_fn = make_simulate_fn(rng=np.random.default_rng(seed + 10_000))
            infer_fn = make_bayesflow_infer_fn(
                workflow.approximator,
                param_key="b_group",
                data_keys=["outcome", "covariate", "group"],
                context_keys={"N": int, "p_alloc": float, "prior_df": float, "prior_scale": float},
            )
            grid_results = run_validation_pipeline(
                conditions_list=conditions,
                n_sims=COARSE_GRID_N_SIMS,
                n_post_draws=COARSE_GRID_N_POST_DRAWS,
                simulate_fn=simulate_fn,
                infer_fn=infer_fn,
                true_param_key="b_arm_treat",
                verbose=False,
            )
            grid_summary = grid_results.get("metrics", {}).get("summary", {})
            grid_mean_cal_error = _safe_float(grid_summary.get("mean_cal_error", np.nan))
            grid_overall_nrmse = _safe_float(grid_summary.get("overall_nrmse", np.nan))
            grid_coverage_90 = _safe_float(grid_summary.get("coverage_90", np.nan))
            grid_coverage_95 = _safe_float(grid_summary.get("coverage_95", np.nan))
        except Exception as grid_exc:
            grid_eval_error = f"{type(grid_exc).__name__}: {grid_exc}"

        history_cal_loss = _safe_float(history.history.get("calibration_loss", [np.nan])[-1])
        history_cal_ema = _safe_float(
            history.history.get("cal_ema", history.history.get("calibration_ema", [np.nan]))[-1]
        )
        history_effective_gamma = _safe_float(history.history.get("effective_gamma", [np.nan])[-1])
        history_gamma = _safe_float(history.history.get("gamma", [np.nan])[-1])
        history_lambda = _safe_float(history.history.get("lambda", [np.nan])[-1])

        final_calibration_ema = history_cal_ema
        if not np.isfinite(final_calibration_ema) and cal_metrics_cb is not None:
            final_calibration_ema = cal_metrics_cb.last_cal_ema

        final_gamma = history_gamma
        if not np.isfinite(final_gamma) and cal_metrics_cb is not None:
            final_gamma = cal_metrics_cb.last_gamma

        final_lambda = history_lambda
        if not np.isfinite(final_lambda) and cal_metrics_cb is not None:
            final_lambda = cal_metrics_cb.last_lambda

        final_calibration_loss = history_cal_loss
        if not np.isfinite(final_calibration_loss):
            final_calibration_loss = final_calibration_ema

        return {
            "variant": name,
            "ok": True,
            "error": "",
            "grid_eval_error": grid_eval_error,
            "use_calibration": use_calibration,
            "gamma_max": float(gamma_max),
            "normalize_loss": bool(normalize_loss) if use_calibration else np.nan,
            "ema_momentum": float(ema_momentum) if use_calibration else np.nan,
            "calibration_mode": float(calibration_mode) if use_calibration else np.nan,
            "schedule_kind": schedule_kind if use_calibration else "baseline",
            "start_epoch": int(start_epoch) if use_calibration else np.nan,
            "end_epoch": int(end_epoch) if use_calibration else np.nan,
            "n_samples": int(n_samples) if use_calibration else np.nan,
            "subsample_size": (np.nan if (not use_calibration or subsample_size is None) else int(subsample_size)),
            "target_calibration_error": float(DEFAULT_TARGET_CALIBRATION_ERROR) if use_calibration else np.nan,
            "lr_lambda": float(DEFAULT_LR_LAMBDA) if use_calibration else np.nan,
            "seed": int(seed),
            "final_train_loss": float(history.history["loss"][-1]),
            "final_val_loss": float(history.history.get("val_loss", [np.nan])[-1]),
            "final_calibration_loss": float(final_calibration_loss),
            "final_calibration_ema": float(final_calibration_ema),
            "final_gamma": float(final_gamma),
            "final_lambda": float(final_lambda),
            "final_effective_gamma": float(history_effective_gamma),
            "nrmse": nrmse,
            "cal_error": cal_error,
            "contraction": contraction,
            "grid_mean_cal_error": float(grid_mean_cal_error),
            "grid_overall_nrmse": float(grid_overall_nrmse),
            "grid_coverage_90": float(grid_coverage_90),
            "grid_coverage_95": float(grid_coverage_95),
        }
    except Exception as exc:
        return {
            "variant": name,
            "ok": False,
            "error": f"{type(exc).__name__}: {exc}",
            "grid_eval_error": "",
            "use_calibration": use_calibration,
            "gamma_max": float(gamma_max),
            "normalize_loss": bool(normalize_loss) if use_calibration else np.nan,
            "ema_momentum": float(ema_momentum) if use_calibration else np.nan,
            "calibration_mode": float(calibration_mode) if use_calibration else np.nan,
            "schedule_kind": schedule_kind if use_calibration else "baseline",
            "start_epoch": int(start_epoch) if use_calibration else np.nan,
            "end_epoch": int(end_epoch) if use_calibration else np.nan,
            "n_samples": int(n_samples) if use_calibration else np.nan,
            "subsample_size": (np.nan if (not use_calibration or subsample_size is None) else int(subsample_size)),
            "target_calibration_error": float(DEFAULT_TARGET_CALIBRATION_ERROR) if use_calibration else np.nan,
            "lr_lambda": float(DEFAULT_LR_LAMBDA) if use_calibration else np.nan,
            "seed": int(seed),
            "final_train_loss": np.nan,
            "final_val_loss": np.nan,
            "final_calibration_loss": np.nan,
            "final_calibration_ema": np.nan,
            "final_gamma": np.nan,
            "final_lambda": np.nan,
            "final_effective_gamma": np.nan,
            "nrmse": np.nan,
            "cal_error": np.nan,
            "contraction": np.nan,
            "grid_mean_cal_error": np.nan,
            "grid_overall_nrmse": np.nan,
            "grid_coverage_90": np.nan,
            "grid_coverage_95": np.nan,
        }

In [ ]:
import gc

def release_backend_memory():
    keras.backend.clear_session()
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass

def _is_oom_message(message):
    msg = str(message).lower()
    return ("outofmemory" in msg) or ("out of memory" in msg) or ("cuda out of memory" in msg)

_original_run_variant = run_variant

def run_variant(name, use_calibration, gamma_max=0.0, calibration_mode=np.nan,
               schedule_kind="windowed_cosine", start_epoch=10, end_epoch=15,
               n_samples=200, subsample_size=None, normalize_loss=True,
               ema_momentum=0.99, seed=2026,
               log_prob_chunk_size=None,
               epochs=COARSE_EPOCHS, batch_size=COARSE_BATCH_SIZE,
               batches_per_epoch=COARSE_BATCHES_PER_EPOCH,
               val_sims=COARSE_VAL_SIMS, test_sims=COARSE_TEST_SIMS):

    release_backend_memory()

    # Apply memory-guided defaults. MEMORY_SETTINGS is set in the memory
    # profiling cell above (suggest_calibration_settings). This replaces the
    # old OOM trial-and-error retry approach with a proactive strategy:
    #   - subsample_size=None -> use the memory-safe default
    #   - n_samples larger than safe limit -> cap and log a notice
    #   - log_prob_chunk_size=None -> enable chunking if the memory check
    #     determined it was needed (i.e. MEMORY_SETTINGS.log_prob_chunk_size
    #     is not None)
    if use_calibration:
        if subsample_size is None:
            subsample_size = MEMORY_SETTINGS.subsample_size
        if n_samples > MEMORY_SETTINGS.n_samples:
            print(f"  [memory] n_samples capped: {n_samples} → {MEMORY_SETTINGS.n_samples}")
            n_samples = MEMORY_SETTINGS.n_samples
        if log_prob_chunk_size is None and MEMORY_SETTINGS.log_prob_chunk_size is not None:
            log_prob_chunk_size = MEMORY_SETTINGS.log_prob_chunk_size

    out = _original_run_variant(
        name=name,
        use_calibration=use_calibration,
        gamma_max=gamma_max,
        calibration_mode=calibration_mode,
        schedule_kind=schedule_kind,
        start_epoch=start_epoch,
        end_epoch=end_epoch,
        n_samples=n_samples,
        subsample_size=subsample_size,
        normalize_loss=normalize_loss,
        ema_momentum=ema_momentum,
        seed=seed,
        log_prob_chunk_size=log_prob_chunk_size,
        epochs=epochs,
        batch_size=batch_size,
        batches_per_epoch=batches_per_epoch,
        val_sims=val_sims,
        test_sims=test_sims,
    )
    out["attempt_batch_size"] = int(batch_size)
    out["attempt_n_samples"] = int(n_samples) if use_calibration else np.nan

    # Minimal OOM safety net: if memory-guided settings still hit OOM (e.g. due
    # to other allocations on the GPU), retry once with halved n_samples and
    # subsample_size before giving up.
    if not out.get("ok", True) and _is_oom_message(out.get("error", "")):
        n_samples_retry = max(32, n_samples // 2)
        subsample_retry = max(16, (subsample_size or 64) // 2)
        print(
            f"  [memory] OOM safety net: retrying with "
            f"n_samples={n_samples_retry}, subsample_size={subsample_retry}"
        )
        release_backend_memory()
        out = _original_run_variant(
            name=name,
            use_calibration=use_calibration,
            gamma_max=gamma_max,
            calibration_mode=calibration_mode,
            schedule_kind=schedule_kind,
            start_epoch=start_epoch,
            end_epoch=end_epoch,
            n_samples=n_samples_retry,
            subsample_size=subsample_retry,
            normalize_loss=normalize_loss,
            ema_momentum=ema_momentum,
            seed=seed,
            log_prob_chunk_size=log_prob_chunk_size,
            epochs=epochs,
            batch_size=batch_size,
            batches_per_epoch=batches_per_epoch,
            val_sims=val_sims,
            test_sims=test_sims,
        )
        out["attempt_batch_size"] = int(batch_size)
        out["attempt_n_samples"] = n_samples_retry
        if not out.get("ok", True):
            out["error"] = "oom_safety_net_failed: " + out.get("error", "")

    return out

In [ ]:
# Principled crossed design: gamma_max × calibration_mode
coarse_variants = [
    dict(name="baseline", use_calibration=False, seed=2026),
]

# Fixed settings for all calibrated runs in this coarse grid
fixed_cfg = dict(
    use_calibration=True,
    schedule_kind="windowed_cosine",
    start_epoch=10,
    end_epoch=15,
    n_samples=200,
    subsample_size=None,  # full sample size (no subsampling)
    seed=2026,
)

# Crossed factors
gamma_grid = [0.5, 1.0, 2.0]
calibration_mode_grid = [ 1.0]

for gamma_max in gamma_grid:
    for calibration_mode in calibration_mode_grid:
        name = f"cosine_g{gamma_max:g}_m{calibration_mode:g}_post200_full"
        coarse_variants.append(
            dict(
                name=name,
                gamma_max=float(gamma_max),
                calibration_mode=float(calibration_mode),
                **fixed_cfg,
            )
        )

print(
    f"Coarse grid size: {len(coarse_variants)} variants "
    f"(1 baseline + {len(gamma_grid)}×{len(calibration_mode_grid)} calibrated)"
)

coarse_rows = []
for cfg in coarse_variants:
    print(f"Running coarse variant: {cfg['name']}")
    coarse_rows.append(run_variant(**cfg))

coarse_df = pd.DataFrame(coarse_rows)
display(coarse_df[[
    "variant", "ok", "schedule_kind", "gamma_max", "calibration_mode",
    "n_samples", "subsample_size",
    "nrmse", "cal_error", "final_val_loss",
    "grid_mean_cal_error", "grid_overall_nrmse", "grid_coverage_95",
    "grid_eval_error", "error",
]])

In [ ]:
coarse_ok = coarse_df.query("ok == True").copy()

baseline_row = coarse_ok.query("variant == 'baseline'").iloc[0]
baseline_nrmse = float(baseline_row["nrmse"] if np.isfinite(baseline_row["nrmse"]) else 1.0)
baseline_grid_nrmse = float(
    baseline_row["grid_overall_nrmse"] if np.isfinite(baseline_row.get("grid_overall_nrmse", np.nan)) else baseline_nrmse
)
nrmse_cap = max(2.0, 2.5 * baseline_nrmse)
grid_nrmse_cap = max(2.0, 2.5 * baseline_grid_nrmse)

# Keep calibrated candidates only and enforce guardrails
coarse_cal = coarse_ok.query("use_calibration == True").copy()
coarse_cal["grid_cov95_abs_err"] = np.abs(coarse_cal["grid_coverage_95"] - 0.95)
coarse_cal["passes_guardrails"] = (
    np.isfinite(coarse_cal["cal_error"])
)
coarse_cal["passes_guardrails"] &= np.isfinite(coarse_cal["nrmse"])
coarse_cal["passes_guardrails"] &= np.isfinite(coarse_cal["final_val_loss"])
coarse_cal["passes_guardrails"] &= np.isfinite(coarse_cal["grid_mean_cal_error"])
coarse_cal["passes_guardrails"] &= np.isfinite(coarse_cal["grid_overall_nrmse"])
coarse_cal["passes_guardrails"] &= np.isfinite(coarse_cal["grid_cov95_abs_err"])
coarse_cal["passes_guardrails"] &= coarse_cal["nrmse"] <= nrmse_cap
coarse_cal["passes_guardrails"] &= coarse_cal["grid_overall_nrmse"] <= grid_nrmse_cap

if coarse_cal["passes_guardrails"].any():
    coarse_pool = coarse_cal.query("passes_guardrails == True").copy()
else:
    print("WARNING: No candidate passed guardrails; using all calibrated candidates.")
    coarse_pool = coarse_cal.copy()

# Final coarse evaluation score is condition-grid based.
# Priority: lower grid calibration error, then grid NRMSE and 95% coverage deviation.
for col in ["grid_mean_cal_error", "grid_overall_nrmse", "grid_cov95_abs_err", "final_val_loss"]:
    cmin, cmax = coarse_pool[col].min(), coarse_pool[col].max()
    if np.isfinite(cmin) and np.isfinite(cmax) and cmax > cmin:
        coarse_pool[f"norm_{col}"] = (coarse_pool[col] - cmin) / (cmax - cmin)
    else:
        coarse_pool[f"norm_{col}"] = 0.0

coarse_pool["score"] = (
    0.55 * coarse_pool["norm_grid_mean_cal_error"]
    + 0.25 * coarse_pool["norm_grid_overall_nrmse"]
    + 0.15 * coarse_pool["norm_grid_cov95_abs_err"]
    + 0.05 * coarse_pool["norm_final_val_loss"]
)

coarse_ranked = coarse_pool.sort_values(
    ["score", "grid_mean_cal_error", "grid_overall_nrmse", "grid_cov95_abs_err"]
).reset_index(drop=True)
print(f"Baseline NRMSE: {baseline_nrmse:.4f}; default guardrail cap: {nrmse_cap:.4f}")
print(f"Baseline grid NRMSE: {baseline_grid_nrmse:.4f}; grid guardrail cap: {grid_nrmse_cap:.4f}")
print("Top coarse candidates (condition-grid ranked):")
display(coarse_ranked[[
    "variant",
    "score",
    "grid_mean_cal_error",
    "grid_overall_nrmse",
    "grid_coverage_95",
    "grid_cov95_abs_err",
    "final_val_loss",
    "cal_error",
    "nrmse",
    "schedule_kind",
    "gamma_max",
    "calibration_mode",
    "n_samples",
    "subsample_size",
]].head(5))

## Phase 2: Refinement of Top Candidates

The next cell re-runs the best coarse candidates with a larger budget and multiple seeds.
This reduces the risk of choosing a brittle setting from one lucky run.

In [ ]:
REFINE_EPOCHS = 50
REFINE_BATCH_SIZE = 256
REFINE_BATCHES_PER_EPOCH = 100
REFINE_VAL_SIMS = 1000
REFINE_TEST_SIMS = 1000
REFINE_SEEDS = [2026, 2027, 2028]
TOP_K = 3

top_candidates = coarse_ranked.head(TOP_K).copy()
refine_rows = []

if top_candidates.empty:
    print("No coarse candidates available for refinement. Skipping refinement runs.")
else:
    for _, row in top_candidates.iterrows():
        base_cfg = row.to_dict()
        for seed in REFINE_SEEDS:
            cfg = {
                "name": f"{base_cfg['variant']}_seed{seed}",
                "use_calibration": bool(base_cfg["use_calibration"]),
                "gamma_max": float(base_cfg["gamma_max"] if np.isfinite(base_cfg["gamma_max"]) else 0.0),
                "calibration_mode": (float(base_cfg["calibration_mode"]) if np.isfinite(base_cfg["calibration_mode"]) else np.nan),
                "schedule_kind": str(base_cfg["schedule_kind"]),
                "start_epoch": int(base_cfg["start_epoch"] if np.isfinite(base_cfg["start_epoch"]) else 0),
                "end_epoch": int(base_cfg["end_epoch"] if np.isfinite(base_cfg["end_epoch"]) else 0),
                "n_samples": int(base_cfg["n_samples"] if np.isfinite(base_cfg["n_samples"]) else 100),
                "subsample_size": (None if not np.isfinite(base_cfg["subsample_size"]) else int(base_cfg["subsample_size"])),
                "seed": seed,
                "epochs": REFINE_EPOCHS,
                "batch_size": REFINE_BATCH_SIZE,
                "batches_per_epoch": REFINE_BATCHES_PER_EPOCH,
                "val_sims": REFINE_VAL_SIMS,
                "test_sims": REFINE_TEST_SIMS,
            }
            print(f"Refinement run: {cfg['name']}")
            out = run_variant(**cfg)
            out["parent_variant"] = base_cfg["variant"]
            refine_rows.append(out)

refine_df = pd.DataFrame(refine_rows)

required_cols = [
    "parent_variant", "variant", "ok", "cal_error", "nrmse", "final_val_loss", "error"
]
for col in required_cols:
    if col not in refine_df.columns:
        refine_df[col] = np.nan

if refine_df.empty:
    print("Refinement produced no runs. Check coarse selection and upstream failures.")

display(refine_df[required_cols])

In [ ]:
ref_ok = refine_df.query("ok == True").copy()

if ref_ok.empty:
    print("No successful refinement runs available.")
    summary = pd.DataFrame()
else:
    # Guardrail at refinement stage: drop unstable high-NRMSE runs
    ref_ok["passes_guardrails"] = ref_ok["nrmse"] <= nrmse_cap
    if ref_ok["passes_guardrails"].any():
        ref_pool = ref_ok.query("passes_guardrails == True").copy()
    else:
        print("WARNING: No refinement run passed NRMSE guardrails; using all successful runs.")
        ref_pool = ref_ok.copy()

    summary = ref_pool.groupby("parent_variant").agg(
        runs=("variant", "count"),
        cal_error_mean=("cal_error", "mean"),
        cal_error_std=("cal_error", "std"),
        nrmse_mean=("nrmse", "mean"),
        nrmse_std=("nrmse", "std"),
        val_loss_mean=("final_val_loss", "mean"),
        val_loss_std=("final_val_loss", "std"),
    ).reset_index()

    # Single-run groups have NaN std; treat as 0 variability for robust scoring.
    summary[["cal_error_std", "nrmse_std", "val_loss_std"]] = summary[[
        "cal_error_std", "nrmse_std", "val_loss_std"
    ]].fillna(0.0)

    for col in ["cal_error_mean", "nrmse_mean", "val_loss_mean", "cal_error_std", "nrmse_std"]:
        cmin, cmax = summary[col].min(), summary[col].max()
        if np.isfinite(cmin) and np.isfinite(cmax) and cmax > cmin:
            summary[f"norm_{col}"] = (summary[col] - cmin) / (cmax - cmin)
        else:
            summary[f"norm_{col}"] = 0.0

    summary["robust_score"] = (
        0.60 * summary["norm_cal_error_mean"]
        + 0.20 * summary["norm_nrmse_mean"]
        + 0.10 * summary["norm_val_loss_mean"]
        + 0.08 * summary["norm_cal_error_std"]
        + 0.02 * summary["norm_nrmse_std"]
    )

    summary = summary.sort_values(["robust_score", "cal_error_mean", "nrmse_mean"]).reset_index(drop=True)

display(summary)

if not summary.empty:
    best_parent = summary.loc[0, "parent_variant"]
    best_match = coarse_ranked.query("variant == @best_parent")
    if best_match.empty:
        best_match = coarse_df.query("variant == @best_parent")
    best_row = best_match.iloc[0]
    print(f"Selected best-practice candidate: {best_parent}")
else:
    print("Falling back to best successful coarse run (no robust refinement summary available).")
    coarse_ok = coarse_df.query("ok == True").copy()
    if coarse_ok.empty:
        raise RuntimeError("No successful coarse or refinement runs available to select best settings.")

    coarse_cal_ok = coarse_ok.query("use_calibration == True").copy()
    fallback_pool = coarse_cal_ok if not coarse_cal_ok.empty else coarse_ok
    best_row = fallback_pool.sort_values(["cal_error", "nrmse", "final_val_loss"]).iloc[0]
    best_parent = str(best_row["variant"])
    print(f"Selected fallback candidate: {best_parent}")

print(best_row[[
    "schedule_kind",
    "gamma_max",
    "calibration_mode",
    "start_epoch",
    "end_epoch",
    "n_samples",
    "subsample_size",
]])

In [ ]:
def finite_or_default(value, default):
    return value if np.isfinite(value) else default

subsample_raw = best_row.get("subsample_size", np.nan)
subsample_value = None if not np.isfinite(subsample_raw) else int(subsample_raw)

normalize_raw = best_row.get("normalize_loss", np.nan)
normalize_value = bool(normalize_raw) if np.isfinite(normalize_raw) else True

cal_mode_raw = best_row.get("calibration_mode", np.nan)
cal_mode_value = float(finite_or_default(cal_mode_raw, 0.0))

# Backward-compatible extraction from historic coarse/refinement tables.
n_post_raw = best_row.get("n_samples", np.nan)
n_post_value = int(finite_or_default(n_post_raw, 200))

start_raw = best_row.get("start_epoch", np.nan)
start_value = int(finite_or_default(start_raw, 6))

end_raw = best_row.get("end_epoch", np.nan)
end_value = int(finite_or_default(end_raw, 14))

gamma_raw = best_row.get("gamma_max", np.nan)
gamma_value = float(finite_or_default(gamma_raw, 0.5))

target_raw = best_row.get("target_calibration_error", np.nan)
target_value = float(finite_or_default(target_raw, DEFAULT_TARGET_CALIBRATION_ERROR))

lr_lambda_raw = best_row.get("lr_lambda", np.nan)
lr_lambda_value = float(finite_or_default(lr_lambda_raw, DEFAULT_LR_LAMBDA))

schedule_value = str(best_row.get("schedule_kind", "windowed_cosine"))
if schedule_value == "baseline":
    schedule_value = "windowed_cosine"

RECOMMENDED_SETTINGS = {
    "schedule_kind": schedule_value,
    "gamma_max": gamma_value,
    "start_epoch": start_value,
    "end_epoch": end_value,
    "normalize_loss": normalize_value,
    "calibration_mode": cal_mode_value,
    "n_samples": n_post_value,
    "subsample_size": subsample_value,
    "target_calibration_error": target_value,
    "lr_lambda": lr_lambda_value,
    "notes": [
        "Lagrangian dual-ascent auto-tunes calibration pressure; gamma_max is used as lambda_max cap.",
        "Run a short coarse sweep before long runs whenever architecture changes.",
        "Use start_epoch warmup to avoid calibration pressure at random-init stage.",
        "Use normalize_loss=True so gamma remains on a stable scale.",
        "Validate on multiple seeds before finalizing model recipe.",
    ],
}

print("Recommended calibration-loss settings:")
display(pd.Series(RECOMMENDED_SETTINGS))

# Optional: persist experiment logs
coarse_df.to_csv("calibration_loss_coarse_sweep.csv", index=False)
refine_df.to_csv("calibration_loss_refinement_runs.csv", index=False)
summary.to_csv("calibration_loss_refinement_summary.csv", index=False)
print("Saved CSV artifacts in notebook working directory")

## Best-Practice Development Notes

1. Start with **warmup + bounded pressure**: tune `start_epoch` and `lambda_max` (stored as `gamma_max` for compatibility in this notebook).
2. Keep first experiments in **conservativeness mode** (`calibration_mode=0.0`) for stability.
3. Use moderate settings first (`lambda_max` around 0.5–2.0, `n_samples` around 80–150, `batch_size_calibration` around 64–96).
4. Always inspect failure traces (`ok=False`) to identify brittle combinations.
5. Choose settings from **multi-seed robust score**, not a single run.

This notebook serves as project documentation for calibration-loss development decisions.

In [ ]:
single_debug = run_variant(
    name="single_debug_g0.1_m0",
    use_calibration=True,
    gamma_max=0.1,
    calibration_mode=0.0,
    schedule_kind="windowed_cosine",
    start_epoch=3,
    end_epoch=6,
    n_samples=100,
    subsample_size=None,
    seed=2026,
    epochs=8,
    batch_size=128,
    batches_per_epoch=10,
    val_sims=200,
    test_sims=200,
)
display(pd.Series(single_debug))

## Calibration Error History (Debug Run)

Run a short calibrated training and plot calibration-error history from the training logs.

In [ ]:
debug_epochs = 8
debug_batch_size = 128
debug_batches_per_epoch = 10
debug_val_sims = 200

def _fit_debug_history(batch_size, n_samples, subsample_size=None):
    release_backend_memory()
    debug_workflow, debug_summary_net, debug_inference_net = build_workflow(
        batch_size, debug_batches_per_epoch
    )

    debug_schedule = build_schedule(
        RECOMMENDED_SETTINGS["schedule_kind"],
        RECOMMENDED_SETTINGS["gamma_max"],
        RECOMMENDED_SETTINGS["start_epoch"],
        RECOMMENDED_SETTINGS["end_epoch"],
    )
    debug_window_epochs = max(1, debug_schedule["end_epoch"] - debug_schedule["start_epoch"] + 1)
    debug_burn_in = max(20, int(0.5 * debug_window_epochs * debug_batches_per_epoch))

    debug_approximator = CalibratedContinuousApproximator(
        inference_network=debug_inference_net,
        summary_network=debug_summary_net,
        adapter=adapter,
        start_epoch=int(debug_schedule["start_epoch"]),
        target_calibration_error=float(RECOMMENDED_SETTINGS["target_calibration_error"]),
        lr_lambda=float(RECOMMENDED_SETTINGS["lr_lambda"]),
        lambda_max=max(1e-6, float(debug_schedule["lambda_max"])),
        normalize_loss=bool(RECOMMENDED_SETTINGS["normalize_loss"]),
        normalization_burn_in=int(debug_burn_in),
        calibration_mode=float(RECOMMENDED_SETTINGS["calibration_mode"]),
        n_samples=int(n_samples),
        batch_size_calibration=subsample_size,
        log_prob_chunk_size=MEMORY_SETTINGS.log_prob_chunk_size,
    )
    debug_approximator.compile(
        optimizer=make_optimizer(batch_size, debug_batches_per_epoch)
    )
    debug_workflow.approximator = debug_approximator

    return debug_workflow.fit_online(
        epochs=debug_epochs,
        batch_size=batch_size,
        num_batches_per_epoch=debug_batches_per_epoch,
        validation_data=debug_val_sims,
        callbacks=[CalibrationMonitorCallback()],
    )

# Use memory-guided settings: cap n_samples at MEMORY_SETTINGS.n_samples and
# resolve subsample_size from RECOMMENDED_SETTINGS or MEMORY_SETTINGS fallback.
debug_n_samples = min(
    int(RECOMMENDED_SETTINGS.get("n_samples", 200)),
    MEMORY_SETTINGS.n_samples,
 )
debug_subsample = (
    RECOMMENDED_SETTINGS.get("subsample_size") or MEMORY_SETTINGS.subsample_size
)

print(
    f"Debug fit: n_samples={debug_n_samples}, subsample_size={debug_subsample}, "
    f"log_prob_chunk_size={MEMORY_SETTINGS.log_prob_chunk_size}"
)

debug_history = _fit_debug_history(
    batch_size=debug_batch_size,
    n_samples=debug_n_samples,
    subsample_size=debug_subsample,
)
print("Debug fit succeeded")

train_cal = debug_history.history.get(
    "calibration_error", debug_history.history.get("calibration_loss", [])
)
val_cal = debug_history.history.get(
    "val_calibration_error", debug_history.history.get("val_calibration_loss", [])
)

fig, ax = plt.subplots(figsize=(8, 4))
if len(train_cal) > 0:
    ax.plot(train_cal, label="Train calibration error", alpha=0.9)
if len(val_cal) > 0:
    ax.plot(val_cal, label="Validation calibration error", alpha=0.9)
ax.set_xlabel("Epoch")
ax.set_ylabel("Calibration Error")
ax.set_title("Calibration Error History (Debug Run)")
if len(train_cal) > 0 or len(val_cal) > 0:
    ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()